# 03 — Tier-1 MCTS-HNM Benchmarks: Multi-Seed Baselines, Hierarchy Ablations & Sensitivity

**Feature**: `F4` (`04-tier1-benchmark-suite` | `EPIC-04`)  
**Architecture Decisions**: `AD-001` (Tier-1 Embedding Protocol), `AD-002` (FastHierarchyTree Protocol), `AD-004` (Scientific Provenance)  

This notebook runs the comprehensive Tier-1 benchmark suite evaluating the MCTS-HNM framework:
1. **Axis 1 (Sampler Comparison)**: Evaluates Uniform, OHEM, Strict MCTS, and Enhanced MCTS across imbalance ratios (Scenario 1, One-vs-Rest).
2. **Axis 2 (Hierarchy Geometry Ablation)**: Compares Semantic WordNet hierarchy vs. Ward confusion tree clustering.
3. **Scenario 2 (Multi-Class Curriculum)**: Evaluates 100-way multi-class classification accuracy, macro-F1, and worst-10-class recall under MCTS curriculum.
4. **Axis 3 (Hyperparameter Sensitivity)**: Grid ablation over exploration parameter $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
5. **Real MCTS Tree Visitation Diagnostics**: Instruments and persists real node visitation counters and superclass distribution to `tier1_visitation_diagnostics.json`.


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.utils.experiment_config import (
    ExecutionMode,
    ExperimentConfig,
    get_isolated_paths,
    print_execution_banner,
)
from src.data.cifar100_taxonomy import get_cifar100_taxonomy
from src.data.extract_embeddings import load_embedding_cache
from src.training.benchmark_runner import (
    audit_scenario1_imbalance,
    run_benchmark,
    run_sensitivity_ablation,
)
from src.utils.seed import set_global_seed

# Execution mode: REAL by default for scientific runs; SMOKE for testing
EXECUTION_MODE = ExecutionMode.REAL
if os.environ.get("EXECUTION_MODE"):
    EXECUTION_MODE = ExecutionMode(os.environ["EXECUTION_MODE"].lower())
elif "pytest" in sys.modules:
    EXECUTION_MODE = ExecutionMode.SMOKE

config = ExperimentConfig(execution_mode=EXECUTION_MODE)
print_execution_banner(config, notebook_name="03_tier1_mcts_benchmarks")
paths = get_isolated_paths(config, repo_root=REPO_ROOT)

SEED = set_global_seed(42)
RESULTS_DIR = paths.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {REPO_ROOT} | Results directory: {RESULTS_DIR}")


EXPERIMENT MODE: REAL
Dataset: CIFAR-100
Synthetic fallback: DISABLED
Scientific results: ENABLED
Notebook: 03_tier1_mcts_benchmarks
Mode: REAL
Output directory: /home/mlima/workspace/mcts-hnm/results/real
Repository root: /home/mlima/workspace/mcts-hnm | Results directory: /home/mlima/workspace/mcts-hnm/results/real


## 1. Load Precomputed ResNet-18 Embeddings

Loads `embeddings/cifar100_resnet18.pt` generated by Notebook 01, or synthesizes a deterministic fixture if cache is absent.

In [2]:
CACHE_PATH = paths.embeddings_dir / "cifar100_resnet18.pt"

if not CACHE_PATH.is_file():
    if config.execution_mode == ExecutionMode.REAL:
        raise FileNotFoundError(
            f"Real CIFAR-100 embeddings not found at {CACHE_PATH}. Run Notebook 01 in REAL mode first."
        )
    else:
        from src.data.extract_embeddings import extract_or_generate_embeddings
        extract_or_generate_embeddings(
            output_path=CACHE_PATH,
            config=config,
            smoke_samples_per_class=30,
            smoke_feature_dim=64,
        )

bundle = load_embedding_cache(CACHE_PATH)
train_features = bundle["train"]["embeddings"]
train_labels = bundle["train"]["fine_labels"]
val_features = bundle["val"]["embeddings"]
val_labels = bundle["val"]["fine_labels"]
test_features = bundle["test"]["embeddings"]
test_labels = bundle["test"]["fine_labels"]
print(f"Loaded {config.execution_mode.value.upper()} embeddings: Train={tuple(train_features.shape)}, Val={tuple(val_features.shape)}, Test={tuple(test_features.shape)}")


Loaded REAL embeddings: Train=(45000, 512), Val=(5000, 512), Test=(10000, 512)


## 2. Axis 1: Sampler Comparison, 2x2 Factorial Ablation & Multi-Ratio Study Expansion (Scenario 1)

Compares negative sampling strategies across multiple seeds on Scenario 1 (One-vs-Rest, imbalanced positive pool $\rho \in \{1:10, 1:50, 1:100\}$ and target classes).

Evaluates the complete **2x2 Factorial Mechanism Grid** to isolate causal effects with computational cost controls:
- `mcts_strict`: Strict MCTS (raw Q-values, 100% focal stream)
- `mcts_qnorm_only`: Min-Max Q-normalization only (100% focal stream)
- `mcts_dualstream_only`: Dual-Stream 50/50 only (raw Q-values)
- `mcts_enhanced`: Enhanced MCTS (Min-Max Q-norm + Dual-Stream 50/50)

Supports multi-ratio expansion (`IMBALANCE_RATIOS_AXIS1`) to evaluate method $\times$ ratio interaction effects, and multi-class generalization (`TARGET_CLASSES_AXIS1`).

In [3]:
SEEDS = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
num_steps = 1000 if config.execution_mode == ExecutionMode.REAL else 10
eval_every = 25 if config.execution_mode == ExecutionMode.REAL else 5
IMBALANCE_RATIOS_AXIS1 = ("1:50",)  # Evaluated canonical ratio; can be expanded to ("1:10", "1:50", "1:100")
TARGET_CLASSES_AXIS1 = (0,)          # Evaluated canonical class (0: beaver); can be expanded to (0, 1, 2)

res_axis1 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "ohem", "mcts_strict", "mcts_qnorm_only", "mcts_dualstream_only", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis1_samplers",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    target_classes=TARGET_CLASSES_AXIS1,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
    imbalance_ratios=IMBALANCE_RATIOS_AXIS1,
)

df_axis1 = pd.read_csv(RESULTS_DIR / "tier1_axis1_samplers.csv")
summary_axis1 = df_axis1.groupby("strategy")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 1: Sampler Comparison Results (Scenario 1) ===")
print(summary_axis1)

# Multi-Ratio / Multi-Class breakdown if expanded
if len(IMBALANCE_RATIOS_AXIS1) > 1 or len(TARGET_CLASSES_AXIS1) > 1:
    group_cols = ["strategy"]
    if len(IMBALANCE_RATIOS_AXIS1) > 1:
        group_cols.append("imbalance_ratio")
    if len(TARGET_CLASSES_AXIS1) > 1:
        group_cols.append("target_class")
    print(f"\n=== Detailed Breakdown by {', '.join(group_cols)} ===")
    print(df_axis1.groupby(group_cols)[["auprc", "f1", "fpr_at_95_recall"]].mean())

# Method x Ratio Interaction Effects
if "ratio_interactions" in res_axis1 and res_axis1["ratio_interactions"]:
    print("\n=== Method x Ratio Interaction Effects ===")
    inter_rows = []
    for pair, comparisons in res_axis1["ratio_interactions"].items():
        for comp, stats in comparisons.items():
            inter_rows.append({
                "ratio_pair": pair,
                "comparison": comp,
                "interaction_mean": stats.get("mean_interaction", 0.0),
                "ci_95_lower": stats.get("ci_95_lower", 0.0),
                "ci_95_upper": stats.get("ci_95_upper", 0.0),
                "cohens_d": stats.get("cohens_d", 0.0),
                "t_pvalue": stats.get("t_pvalue"),
            })
    if inter_rows:
        print(pd.DataFrame(inter_rows).to_string(index=False))

# Computational Cost Summary
cost_cols = ["total_updates", "gradient_samples", "mining_forwards", "elapsed_seconds"]
present_cost_cols = [c for c in cost_cols if c in df_axis1.columns]
if present_cost_cols:
    print("\n=== Axis 1: Computational Cost Controls ===")
    print(df_axis1.groupby("strategy")[present_cost_cols].mean())

# 2x2 Factorial Mechanism Decomposition
if "factorial_2x2_effects" in res_axis1 and "semantic" in res_axis1["factorial_2x2_effects"]:
    effects = res_axis1["factorial_2x2_effects"]["semantic"]
    print("\n=== 2x2 Factorial Mechanism Decomposition (AUPRC) ===")
    p_eff = effects["primary_effects"]
    q_ci = p_eff["main_effect_qnorm"]["ci_95"]
    ds_ci = p_eff["main_effect_dualstream"]["ci_95"]
    int_ci = p_eff["interaction_effect"]["ci_95"]
    print(f"  Main Effect Q-Norm:      {p_eff['main_effect_qnorm']['mean']:+.4f} (95% CI: [{q_ci[0]:+.4f}, {q_ci[1]:+.4f}])")
    print(f"  Main Effect Dual-Stream: {p_eff['main_effect_dualstream']['mean']:+.4f} (95% CI: [{ds_ci[0]:+.4f}, {ds_ci[1]:+.4f}])")
    print(f"  Interaction Term:        {p_eff['interaction_effect']['mean']:+.4f} (95% CI: [{int_ci[0]:+.4f}, {int_ci[1]:+.4f}])")

if res_axis1.get("statistical_comparisons"):
    print("\nStatistical tests against Enhanced MCTS-HNM:")
    for comp, stats in res_axis1["statistical_comparisons"].items():
        print(f"  {comp}: diff={stats['mean_diff']:.4f}, t_p={stats['t_pvalue']:.4e}, w_p={stats['wilcoxon_pvalue']:.4e}")


=== Axis 1: Sampler Comparison Results (Scenario 1) ===
                         auprc                  f1           fpr_at_95_recall  \
                          mean       std      mean       std             mean   
strategy                                                                        
mcts_dualstream_only  0.733582  0.018397  0.711145  0.015412         0.286242   
mcts_enhanced         0.684816  0.022518  0.679669  0.030629         0.355737   
mcts_qnorm_only       0.683815  0.010098  0.655708  0.040458         0.331374   
mcts_strict           0.735776  0.016832  0.696937  0.017994         0.240303   
ohem                  0.703195  0.012520  0.669803  0.012659         0.326364   
uniform               0.707786  0.038021  0.685252  0.026620         0.282202   

                                
                           std  
strategy                        
mcts_dualstream_only  0.145203  
mcts_enhanced         0.169167  
mcts_qnorm_only       0.151736  
mcts_strict     

## 3. Axis 2: Hierarchy Geometry Ablation & Paired Inferential Testing (Semantic vs. Ward Confusion Tree)

Tests whether empirical visual error clustering (Ward linkage on validation confusion matrix) improves MCTS mining efficiency over static taxonomy.
Extracts structural tree topology diagnostics and conducts seed-by-seed paired statistical tests (paired t-test, Wilcoxon, Cohen's d, 95% CI) to formally assess the null hypothesis of equivalence.

In [4]:
res_axis2 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("mcts_enhanced",),
    tree_types=("semantic", "confusion_ward"),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis2_tree_ablation",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_axis2 = pd.read_csv(RESULTS_DIR / "tier1_axis2_tree_ablation.csv")
summary_axis2 = df_axis2.groupby("tree_type")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 2: Tree Geometry Ablation Results ===")
print(summary_axis2)

# Tree Topological Diagnostics
if "tree_diagnostics" in res_axis2:
    print("\n=== Tree Topology Diagnostics ===")
    diag_df = pd.DataFrame(res_axis2["tree_diagnostics"]).T
    diag_cols = ["max_depth", "mean_leaf_depth", "total_nodes", "num_leaves", "branching_factor_mean"]
    print(diag_df[[c for c in diag_cols if c in diag_df.columns]])

# Paired Inferential Statistical Comparison (Semantic vs Ward)
if "paired_hierarchy_tests" in res_axis2 and res_axis2["paired_hierarchy_tests"]:
    pht = res_axis2["paired_hierarchy_tests"]
    strat_key = "mcts_enhanced" if "mcts_enhanced" in pht else [k for k in pht.keys() if k != "verdict"][0]
    entry = pht[strat_key]
    print(f"\n=== Paired Inferential Comparison: Semantic vs. Ward Confusion Tree ({strat_key}) ===")
    p_stats = entry.get("primary_stats", {})
    f1_stats = entry.get("f1_stats", {})
    rows = [
        {
            "metric": "AUPRC (Primary)",
            "delta_mean (Sem - Ward)": entry.get("delta_primary_mean", 0.0),
            "ci_95_lower": p_stats.get("ci_95_lower", 0.0),
            "ci_95_upper": p_stats.get("ci_95_upper", 0.0),
            "cohens_d": p_stats.get("cohens_d", 0.0),
            "t_pvalue": p_stats.get("t_pvalue"),
            "wilcoxon_pvalue": p_stats.get("wilcoxon_pvalue"),
        },
        {
            "metric": "F1",
            "delta_mean (Sem - Ward)": entry.get("delta_f1_mean", 0.0),
            "ci_95_lower": f1_stats.get("ci_95_lower", 0.0),
            "ci_95_upper": f1_stats.get("ci_95_upper", 0.0),
            "cohens_d": f1_stats.get("cohens_d", 0.0),
            "t_pvalue": f1_stats.get("t_pvalue"),
            "wilcoxon_pvalue": f1_stats.get("wilcoxon_pvalue"),
        },
    ]
    print(pd.DataFrame(rows).to_string(index=False))
    verdict = entry.get("verdict") or pht.get("verdict", "No verdict computed.")
    print("\n=== Formal Statistical Verdict ===")
    print(f"  {verdict}")


=== Axis 2: Tree Geometry Ablation Results ===
                   auprc                  f1           fpr_at_95_recall  \
                    mean       std      mean       std             mean   
tree_type                                                                 
confusion_ward  0.674910  0.040757  0.646926  0.050195         0.329051   
semantic        0.684816  0.022518  0.679669  0.030629         0.355737   

                          
                     std  
tree_type                 
confusion_ward  0.197107  
semantic        0.169167  

=== Tree Topology Diagnostics ===
               max_depth mean_leaf_depth total_nodes num_leaves  \
semantic               2             2.0         120         99   
confusion_ward        39       17.454545         197         99   

               branching_factor_mean  
semantic                    5.666667  
confusion_ward                   2.0  

=== Paired Inferential Comparison: Semantic vs. Ward Confusion Tree (mcts_enhanced) ===

## 4. Scenario 2: Multi-Class Curriculum Benchmark & Causal Coverage Diagnostics

Evaluates 100-way multi-class classification accuracy, macro-F1, worst-10-class recall, and coverage diagnostics across 4 key strategies:
- `uniform`: Standard uniform sampling baseline (64 uniform samples)
- `uniform_grouped`: Homogeneity control (32 random focal + 32 uniform samples)
- `mcts_enhanced`: Historical MCTS baseline (Min-Max Q-norm, 32 focal + 32 uniform samples)
- `mcts_coverage_balanced`: Corrective balanced policy (Min-Max Q-norm, 16 focal + 48 uniform samples)


In [5]:
s2_seeds = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
s2_steps = 2000 if config.execution_mode == ExecutionMode.REAL else 10
s2_eval = 20 if config.execution_mode == ExecutionMode.REAL else 5

res_s2 = run_benchmark(
    scenario="scenario2",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "uniform_grouped", "mcts_enhanced", "mcts_coverage_balanced"),
    tree_types=("semantic",),
    seeds=s2_seeds,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_scenario2_curriculum",
    num_steps=s2_steps,
    eval_every=s2_eval,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_s2 = pd.read_csv(RESULTS_DIR / "tier1_scenario2_curriculum.csv")
print("=== Scenario 2: Multi-Class Curriculum Results ===")
summary_cols = ["top1_accuracy", "macro_f1", "worst_10_class_recall", "exposure_entropy", "class_coverage", "unvisited_classes", "exposure_recall_correlation"]
summary_s2 = df_s2.groupby("strategy")[[c for c in summary_cols if c in df_s2.columns]].agg(["mean", "std"])
print(summary_s2)

# Paired Differences vs Uniform Baseline
if "paired_differences_against_uniform" in res_s2 and res_s2["paired_differences_against_uniform"]:
    print("\n=== Paired Differences Against Uniform Baseline ===")
    for strat, diff_data in res_s2["paired_differences_against_uniform"].items():
        delta_acc = diff_data.get("delta_primary_mean", 0.0) * 100.0
        delta_f1 = diff_data.get("delta_f1_mean", 0.0) * 100.0
        delta_w10 = diff_data.get("delta_worst_10_mean", 0.0) * 100.0
        print(f"[{strat} vs Uniform] ΔAcc: {delta_acc:+.2f} pp | ΔMacro-F1: {delta_f1:+.2f} pp | ΔWorst10: {delta_w10:+.2f} pp")

print("\n=== Causal Inference Diagnostics ===")
mean_metrics = df_s2.groupby("strategy")[[c for c in summary_cols if c in df_s2.columns]].mean()
uni_acc = mean_metrics.loc["uniform", "top1_accuracy"] if "uniform" in mean_metrics.index else 0.0
grp_acc = mean_metrics.loc["uniform_grouped", "top1_accuracy"] if "uniform_grouped" in mean_metrics.index else 0.0
enh_acc = mean_metrics.loc["mcts_enhanced", "top1_accuracy"] if "mcts_enhanced" in mean_metrics.index else 0.0
bal_acc = mean_metrics.loc["mcts_coverage_balanced", "top1_accuracy"] if "mcts_coverage_balanced" in mean_metrics.index else 0.0

if grp_acc < uni_acc - 0.10:
    print("  -> Batch Homogeneity Effect CONFIRMED: UniformGrouped (32 focal / 32 uniform) degrades significantly,")
    print("     demonstrating that label saturation in the minibatch strongly drives multi-class performance loss.")
else:
    print("  -> Batch homogeneity alone does not explain degradation; MCTS tree exploration policy contributes.")

if bal_acc > enh_acc:
    print(f"  -> MCTS Coverage Balanced (16 focal / 48 uniform) improved Top-1 Acc by {bal_acc - enh_acc:.4f} over Enhanced,")
    print("     demonstrating that mitigating focal ratio preserves class diversity while retaining hierarchical mining signal.")


=== Scenario 2: Multi-Class Curriculum Results ===
                       top1_accuracy            macro_f1            \
                                mean       std      mean       std   
strategy                                                             
mcts_coverage_balanced       0.53058  0.010518  0.543620  0.009476   
mcts_enhanced                0.33434  0.015394  0.342589  0.013655   
uniform                      0.63854  0.004351  0.637248  0.004391   
uniform_grouped              0.52240  0.009672  0.514411  0.011732   

                       worst_10_class_recall           exposure_entropy  \
                                        mean       std             mean   
strategy                                                                  
mcts_coverage_balanced                0.1024  0.036011         0.995092   
mcts_enhanced                         0.0068  0.004817         0.986598   
uniform                               0.3346  0.014536         0.999905   
uniform_

## 5. Axis 3: MCTS Hyperparameter Sensitivity Ablation ($c$ and $\alpha$)

Evaluates the sensitivity of MCTS-HNM to the exploration coefficient $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum factor $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
Uses identical `build_scenario1_split` partitioning with `imbalance_ratio="1:50"` (effective pool ratio ~1:4,950, ~9 training positives) and paired seeds matching Axis 1.
Results are exported to `results/<mode>/tier1_hyperparameter_sensitivity.json` and `.csv` reporting the 4-quadrant evaluation policy (last vs. best_val x 0.5 vs. calibrated thresholds).


In [6]:
c_vals = (0.5, 1.0, 2.0, 4.0) if config.execution_mode == ExecutionMode.REAL else (0.5, 1.0)
sens_seeds = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
alpha_vals = (0.1, 0.3, 0.5, 0.7) if config.execution_mode == ExecutionMode.REAL else (0.1, 0.3)
sens_steps = 1000 if config.execution_mode == ExecutionMode.REAL else 10
sens_eval = 50 if config.execution_mode == ExecutionMode.REAL else 5

res_sens = run_sensitivity_ablation(
    train_features=train_features,
    train_labels=train_labels,
    val_features=val_features,
    val_labels=val_labels,
    test_features=test_features,
    test_labels=test_labels,
    c_values=c_vals,
    alpha_values=alpha_vals,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_hyperparameter_sensitivity",
    imbalance_ratio="1:50",
    num_steps=sens_steps,
    eval_every=sens_eval,
    seeds=sens_seeds,
    config=config,
)

print("=== Axis 3: Hyperparameter Sensitivity Grid Completed ===")
print(f"Imbalance ratio: {res_sens.get('imbalance_ratio')} | Train pool: {res_sens.get('num_positives_train')} pos / {res_sens.get('num_negatives_train')} neg")
df_sens = pd.DataFrame(res_sens["grid"])
display_cols = ["c", "alpha", "f1_best_val_calibrated_mean", "f1_best_val_thresh05_mean", "f1_last_calibrated_mean", "f1_last_thresh05_mean", "auprc_mean"]
print(df_sens[[col for col in display_cols if col in df_sens.columns]])


=== Axis 3: Hyperparameter Sensitivity Grid Completed ===
Imbalance ratio: 1:50 | Train pool: 9 pos / 44550 neg
      c  alpha  f1_best_val_calibrated_mean  f1_best_val_thresh05_mean  \
0   0.5    0.1                     0.700050                   0.668644   
1   0.5    0.3                     0.701949                   0.681394   
2   0.5    0.5                     0.713626                   0.665936   
3   0.5    0.7                     0.702896                   0.677900   
4   1.0    0.1                     0.697521                   0.643802   
5   1.0    0.3                     0.695518                   0.666017   
6   1.0    0.5                     0.692211                   0.682204   
7   1.0    0.7                     0.710007                   0.683035   
8   2.0    0.1                     0.698858                   0.674344   
9   2.0    0.3                     0.699433                   0.677709   
10  2.0    0.5                     0.709140                   0.684513   


## 6. Summary & Verification

All Tier-1 multi-seed experimental outputs and visitation diagnostics have been saved to disk under `results/` for figure and table generation in Notebook 05 (`F6`).


In [7]:
print("Exported files in results/:")
for f in sorted(RESULTS_DIR.glob("tier1_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print("Tier-1 MCTS-HNM benchmark execution complete.")

Exported files in results/:
  tier1_axis1_samplers.csv (8684 bytes)
  tier1_axis1_samplers.csv.meta.json (302 bytes)
  tier1_axis1_samplers.json (330615 bytes)
  tier1_axis1_samplers.json.meta.json (302 bytes)
  tier1_axis2_tree_ablation.csv (3125 bytes)
  tier1_axis2_tree_ablation.csv.meta.json (302 bytes)
  tier1_axis2_tree_ablation.json (248913 bytes)
  tier1_axis2_tree_ablation.json.meta.json (302 bytes)
  tier1_hyperparameter_sensitivity.csv (23905 bytes)
  tier1_hyperparameter_sensitivity.csv.meta.json (302 bytes)
  tier1_hyperparameter_sensitivity.json (99550 bytes)
  tier1_hyperparameter_sensitivity.json.meta.json (302 bytes)
  tier1_scenario2_curriculum.csv (4053 bytes)
  tier1_scenario2_curriculum.csv.meta.json (302 bytes)
  tier1_scenario2_curriculum.json (699175 bytes)
  tier1_scenario2_curriculum.json.meta.json (302 bytes)
  tier1_scenario2_diagnostics.json (31691 bytes)
  tier1_scenario2_diagnostics.json.meta.json (302 bytes)
  tier1_visitation_diagnostics.json (461479 by